# RPLAN Inventory and SCI Split Workflow

This notebook is the first serious gate before training. It inventories the RPLAN folder on Google Drive, converts the dataset to the canonical SCI JSONL format, validates that JSONL, and freezes train/validation/test splits.

Do not tune models on the test split after running this workflow.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'SASA-GAN_Buildings'
RPLAN_DIR = DRIVE_ROOT / 'RPLAN'  # TODO: change to your actual RPLAN folder

SCI = PROJECT_DIR / 'sci_system'
METADATA_DIR = PROJECT_DIR / 'metadata'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
REPORT_DIR = SCI / 'reports'

for p in [METADATA_DIR, OUTPUT_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR =', PROJECT_DIR)
print('RPLAN_DIR   =', RPLAN_DIR)

## 1. Inventory the Drive Dataset

Run this first. Send `rplan_drive_inventory.json` back for converter hardening if the template does not work directly.

In [ ]:
!python "$SCI/colab/inspect_drive_dataset.py" \
  --root "$RPLAN_DIR" \
  --output-json "$REPORT_DIR/rplan_drive_inventory.json"

## 2. Convert RPLAN to Canonical JSONL

Set `SOURCE_FILE` to the actual data file identified in the inventory. If conversion fails, inspect the inventory and edit `rplan_to_sci_jsonl_template.py`.

In [ ]:
SOURCE_FILE = RPLAN_DIR / 'YOUR_SOURCE_FILE.pkl'  # TODO: change this after inventory
PLANS_JSONL = METADATA_DIR / 'plans.jsonl'

!python "$SCI/colab/rplan_to_sci_jsonl_template.py" \
  --source "$SOURCE_FILE" \
  --output "$PLANS_JSONL"

## 3. Validate JSONL Schema

This must pass before split generation.

In [ ]:
!python "$SCI/scripts/validate_layout_jsonl.py" \
  --input "$PLANS_JSONL" \
  --output-json "$REPORT_DIR/plans_schema_validation.json"

## 4. Freeze Train/Validation/Test Split

After this point, use validation for tuning and test only for final reporting.

In [ ]:
!python "$SCI/scripts/make_splits.py" \
  --input "$PLANS_JSONL" \
  --output-dir "$REPORT_DIR/splits" \
  --group-key family_id \
  --seed 20260610

## 5. Next Outputs Needed

After model training, export:

- `outputs/test_ground_truth.jsonl`
- `outputs/predictions/MODEL_NAME_test.jsonl`
- `outputs/dxf_test/`

Then run the metric, architectural-screening, and DXF-validation commands from `COLAB_RPLAN_WORKFLOW.md`.